# Lesson 03 - Agentic Design Patterns

এই পাঠে, আমরা কার্যকর AI এজেন্ট তৈরি করার জন্য তিনটি ভিত্তিপ্রস্তর ডিজাইন প্যাটার্ন অন্বেষণ করব:

1. **স্পষ্ট এজেন্ট নির্দেশনা** — এজেন্টের আচরণ পরিচালনা করার জন্য সুনির্দিষ্ট, ভূমিকা-সংজ্ঞায়িত প্রম্পট তৈরি করা
2. **Pydantic মডেলের মাধ্যমে কাঠামোবদ্ধ আউটপুট** — নিশ্চিত করা যে এজেন্টগুলি পূর্বানুমানযোগ্য, যাচাইকৃত ডেটা প্রদান করে
3. **একক দায়িত্ব এজেন্ট** — এমন ফোকাসড এজেন্ট ডিজাইন করা যা প্রতিটি একটি কাজ ভালোভাবে করে

আমরা প্রতি প্যাটার্ন একটি **ভ্রমণ গন্তব্যের সুপারিশকারী** পরিস্থিতিতে প্রয়োগ করব, ধাপে ধাপে এমন একটি সিস্টেম তৈরি করব যা গন্তব্য সুপারিশ করতে পারে, উপলব্ধতা পরীক্ষা করতে পারে, এবং লজিস্টিকস পরিচালনা করতে পারে।


## সেটআপ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## প্যাটার্ন ১: স্পষ্ট এজেন্ট নির্দেশিকা

সবচেয়ে প্রভাবশালী প্যাটার্নটিও সবচেয়ে সহজ: আপনার এজেন্টের জন্য স্পষ্ট, বিস্তারিত নির্দেশিকা লেখা।

ভাল নির্দেশনাগুলো নির্ধারণ করে:
- **কে** এজেন্টটি (ব্যক্তিত্ব এবং টোন)
- **কি** এটি করা উচিত (ধাপে ধাপে দায়িত্ব)
- **কিভাবে** এটি আচরণ করা উচিত (সীমাবদ্ধতা এবং শৈলী)

নিচে, আমরা একটি ট্রাভেল কনসিয়ার্জ এজেন্ট তৈরি করছি স্পষ্ট নির্দেশনাসহ যা প্রতিটি প্রতিক্রিয়া গঠন করে যা এটি তৈরি করে।


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Pattern 2: Pydantic মডেল সহ কাঠামোবদ্ধ আউটপুট

স্বচ্ছন্দ লেখাটি কথোপকথনের জন্য কার্যকর, তবে পরবর্তী সিস্টেমে কাঠামোবদ্ধ ডেটার প্রয়োজন হয়।
**Pydantic মডেল** এবং একটি **টুল ফাংশন** এর যুগলবন্দী করে, আমরা:

- এজেন্টের আউটপুটের জন্য একটি নির্দিষ্ট স্কিমা সংজ্ঞায়িত করতে পারি
- স্বয়ংক্রিয়ভাবে প্রতিক্রিয়া যাচাই করতে পারি
- অ্যাপ্লিকেশনের লজিকে এজেন্ট ফলাফল নির্ভরযোগ্যভাবে সংহত করতে পারি

আমরা একটি টুলও পরিচয় করিয়ে দেই যা গন্তব্যের বিবরণ প্রদান করে যাতে এজেন্ট তার সুপারিশগুলি বাস্তব তথ্যের উপর ভিত্তি করে।


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## প্যাটার্ন ৩: একক দায়িত্ব এজেন্টরা

জটিল কাজগুলি লাভবান হয় একাধিক লক্ষ্যভিত্তিক এজেন্টের মধ্যে কাজ ভাগ করে নেওয়ার মাধ্যমে, যার প্রত্যেকটির একটি একক দায়িত্ব থাকে:

- একটি **গন্তব্য বিশেষজ্ঞ** যিনি জায়গা এবং উপলভ্যতা সম্পর্কে জানেন
- একটি **লজিস্টিক পরিকল্পনাকারী** যিনি ফ্লাইট, হোটেল, এবং যাত্রাপথ পরিচালনা করেন

এটি সফটওয়্যার ইঞ্জিনিয়ারিংয়ের *চিন্তার পৃথকীকরণ* নীতির সামঞ্জস্যপূর্ণ — প্রতিটি এজেন্টকে স্বাধীনভাবে পরীক্ষা, রক্ষণাবেক্ষণ, এবং উন্নত করা সহজ হয়।


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## সারসংক্ষেপ

এই লেসনে আমরা একটি ট্রাভেল রিকমেন্ডার দৃশ্যের জন্য তিনটি এজেন্টিক ডিজাইন প্যাটার্ন প্রয়োগ করেছি:

| প্যাটার্ন | মূল ধারণা | সুবিধা |
|---|---|---|
| **স্বচ্ছ নির্দেশাবলী** | পাত্র, দায়িত্ব, এবং সীমাবদ্ধতা শুরুতেই নির্ধারণ করুন | সঙ্গতিপূর্ণ, ব্র্যান্ড-সঙ্গত এজেন্ট আচরণ |
| **গঠনমূলক আউটপুট** | উত্তরের ফরম্যাট হিসেবে Pydantic মডেল ব্যবহার করুন | যাচাইকৃত, মেশিনে পড়ার যোগ্য ফলাফল |
| **একক দায়িত্ব** | প্রত্যেক এজেন্টকে একটি নির্দিষ্ট কাজ দিন | সহজে পরীক্ষা, রক্ষণাবেক্ষণ, এবং সংমিশ্রণ করা যায় |

এই প্যাটার্নগুলি স্বাভাবিকভাবে সংমিশ্রিত হয় — আপনি স্পষ্ট নির্দেশাবলী এবং গঠনমূলক আউটপুটকে একক দায়িত্বের এজেন্টের মধ্যে মিলিয়ে শক্তপোক্ত, প্রোডাকশন-রেডি সিস্টেম তৈরি করতে পারেন।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
